# Assignment 4: Retrieval-Augmented Generation

*Published April 27, 2026*

---

## Pedagogical Purposes

Students will:
- Understand RAG applications in NLP
- Learn LangChain for building NLP applications
- Recognize RAG challenges and use cases

---

## Requirements

- Optional feedback submission via Canvas
- **Deadline:** May 28
- Submit Colab notebook, Github repo, or Python files
- Include a document indicating which tasks need feedback
- Programming-focused (no technical report required)
- Oral exam during course conclusion covering a subset of tasks

## Preliminaries

Install the required packages:

In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install langchain-core
!pip install sentence_transformers
!pip install langchain-chroma

---

## Part 1: The Dataset

### Task 1.1 — Download PubMedQA Dataset

Download the [PubMedQA dataset](https://pubmedqa.github.io/) based on medical research abstracts:

In [ ]:
!wget https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json

### Collect Two Datasets

Process the downloaded file into questions and documents:

In [ ]:
import pandas as pd

tmp_data = pd.read_json("ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({
    "abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS + [row.LONG_ANSWER]), axis=1),
    "year": tmp_data.YEAR
})

questions = pd.DataFrame({
    "question": tmp_data.QUESTION,
    "year": tmp_data.YEAR,
    "gold_label": tmp_data.final_decision,
    "gold_context": tmp_data.LONG_ANSWER,
    "gold_document_id": documents.index
})

**Sanity check:** Inspect a sample question and document:

In [ ]:
print(questions.iloc[0].question)
print(documents.iloc[0].abstract)

---

## Part 2: Configure LangChain LM

### Task 2.1 — Select a Language Model

Browse HuggingFace models and load one using `HuggingFacePipeline.from_model_id`.  
Set `return_full_text=False` and invoke the model with `model.invoke(your_prompt)`.

> **Note:** Gated models require a HuggingFace account and a token with *"Read access to contents of all public gated repos."*

**Sanity check:** Verify the model returns reasonable output.

In [ ]:
# Task 2.1 — Load a language model from HuggingFace


---

## Part 3: Set Up Document Database

### Task 3.1 — Embedding Model

Use the `HuggingFaceEmbeddings` function to load an embedding model.  
Call `embed_query` and verify the output shape is `(embedding_dim,)`.

In [ ]:
# Task 3.1 — Load an embedding model and verify output shape


### Task 3.2 — Chunking

Use `RecursiveCharacterTextSplitter` to chunk `documents.abstract`.  
Create LangChain `Document` objects with metadata:

In [ ]:
# Task 3.2 — Initialize the text splitter with your chosen chunk size / overlap
# text_splitter = RecursiveCharacterTextSplitter(...)

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(texts=documents.abstract.tolist(), metadata=metadatas)

> **Reflection:** How do chunking design choices (chunk size, overlap, splitting strategy) affect RAG quality?

### Task 3.3 — Vector Store

Use Chroma with cosine similarity.  
Apply `Chroma.from_documents` or `vector_store.add_documents` to index the chunks.  
Test retrieval with:

In [ ]:
# Task 3.3 — Build the Chroma vector store

# vector_store = Chroma.from_documents(...)


In [ ]:
# Sanity check — test similarity search
results = vector_store.similarity_search_with_score(
    "What is programmed cell death?", k=3
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

---

## Part 4: Implementing the System

### Task 4.1 — Full RAG Pipeline

Choose **Option A** or **Option B** (not both).

---

### Option A: Agent-Based RAG

Create custom middleware that retrieves documents before each model call:

In [ ]:
from typing import Any
from langchain_core.documents import Document
from langchain.agents.middleware import AgentMiddleware, AgentState

class State(AgentState):
    context: list[Document]

class RetrieveDocumentsMiddleware(AgentMiddleware[State]):
    state_schema = State

    def __init__(self, vector_store):
        self.vector_store = vector_store

    def before_model(self, state: AgentState) -> dict[str, Any] | None:
        last_message = state["messages"][-1]
        retrieved_docs = self.vector_store.similarity_search(last_message.text)
        docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
        augmented_message_content = (
            # Put your prompt here
        )
        return {
            "messages": [last_message.model_copy(update={"content": augmented_message_content})],
            "context": retrieved_docs,
        }

> **Hint:** Craft a prompt that instructs the model to answer yes/no classification questions.

Create an agent with `create_agent` using the middleware, then stream results:

In [ ]:
# Task 4.1 Option A — Create agent and test it

# agent = create_agent(..., middleware=[RetrieveDocumentsMiddleware(vector_store)])

your_query = questions.iloc[0].question

for step in agent.stream(
    {"messages": [{"role": "user", "content": your_query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

---

### Option B: Chain-Based RAG (LCEL)

Define a retriever from the vector store and build a chain using LangChain Expression Language:

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel

# Task 4.1 Option B — Build the RAG chain

retriever = vector_store.as_retriever()

# Define your prompt template
# prompt = ChatPromptTemplate.from_template(...)

# Build the chain
# chain = (
#     prompt
#     | model
#     | StrOutputParser()
# )

# Combine retriever and chain using RunnableParallel
# rag_chain = runnable_parallel_object.assign(answer=chain)


**Sanity check:** Test with a sample question and verify document retrieval and answer quality:

In [ ]:
# Sanity check — run the chain on a sample question
sample_question = questions.iloc[0].question

# answer = rag_chain.invoke({"question": sample_question})
# print(answer["answer"])
# print(answer["context"])  # retrieved documents


---

## Part 5: Evaluate RAG

### Task 5.1 — High-Level Evaluation

Evaluate your system on `questions.question` and `questions.gold_label` using F1 and/or accuracy metrics.  
Handle invalid/unparseable model answers separately.  
Compare RAG performance against a baseline LM **without** context.

In [ ]:
# Task 5.1 — Evaluate RAG vs. baseline LM


### Task 5.2 — Detailed Inspection

Compare retrieved document IDs against `questions.gold_document_id`.  
Inspect samples and assess how well the pipeline is functioning.

In [ ]:
# Task 5.2 — Compare retrieved document IDs to gold document IDs
